# Model Evaluation & Comparison

**Deliverable 04** - Part 1

**Author:** Abdelrahman Yasser Hassan Zaky  
**ID:** 231000102

## Step 1: Load All Models & Test Data

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import cross_val_score

with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
X_train_scaled = data['X_train_scaled']
X_test_scaled = data['X_test_scaled']
le_type = data['le_type']
type_classes = le_type.classes_

with open('../data/knn_model.pkl', 'rb') as f:
    knn = pickle.load(f)
with open('../data/decision_tree_model.pkl', 'rb') as f:
    dt = pickle.load(f)
with open('../data/random_forest_model.pkl', 'rb') as f:
    rf = pickle.load(f)
with open('../data/naive_bayes_model.pkl', 'rb') as f:
    nb = pickle.load(f)
with open('../data/svm_model.pkl', 'rb') as f:
    svm = pickle.load(f)

print('All models loaded successfully')
print(f'Test set: {len(X_test)} samples')

## Step 2: Evaluate All Models

In [ ]:
models = {
    'KNN': (knn, X_test_scaled),
    'Decision Tree': (dt, X_test),
    'Random Forest': (rf, X_test),
    'Naive Bayes': (nb, X_test_scaled),
    'SVM': (svm, X_test_scaled),
}

results = {}
for name, (model, X) in models.items():
    y_pred = model.predict(X)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    results[name] = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
    print(f'{name}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}')

## Step 3: Create Comparison Table & Visualizations

In [ ]:
import pandas as pd

df_results = pd.DataFrame(results).T
df_results.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
df_results.index.name = 'Model'
print(df_results.round(4).to_string())

In [ ]:
model_names = list(results.keys())
accs = [results[m]['accuracy'] for m in model_names]

plt.figure(figsize=(10, 6))
bars = plt.bar(model_names, accs, color=['#3498db', '#2ecc71', '#27ae60', '#e74c3c', '#9b59b6'])
plt.title('Model Accuracy Comparison', fontsize=14)
plt.xlabel('Model', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.ylim(0, 1)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.2f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/accuracy_comparison.png', dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))
x = np.arange(len(model_names))
width = 0.2
metrics = ['accuracy', 'precision', 'recall', 'f1']
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
for i, metric in enumerate(metrics):
    vals = [results[m][metric] for m in model_names]
    plt.bar(x + i*width, vals, width, label=metric.title(), color=colors[i])
plt.xlabel('Model', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Multi-Metric Model Comparison', fontsize=14)
plt.xticks(x + width*1.5, model_names)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig('../reports/metrics_comparison.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for idx, (name, (model, X)) in enumerate(models.items()):
    y_pred = model.predict(X)
    cm = confusion_matrix(y_test, y_pred)
    im = axes[idx].imshow(cm, interpolation='nearest', cmap='Blues')
    axes[idx].set_title(name, fontsize=12)
    tick_marks = np.arange(len(type_classes))
    axes[idx].set_xticks(tick_marks)
    axes[idx].set_xticklabels(type_classes, rotation=45, ha='right', fontsize=8)
    axes[idx].set_yticks(tick_marks)
    axes[idx].set_yticklabels(type_classes, fontsize=8)
    axes[idx].set_ylabel('True', fontsize=10)
    axes[idx].set_xlabel('Pred', fontsize=10)
    for i in range(len(type_classes)):
        for j in range(len(type_classes)):
            color = 'red' if cm[i, j] > cm.max() / 2 else 'black'
            axes[idx].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=7, color=color)
plt.tight_layout()
plt.savefig('../reports/confusion_matrices.png', dpi=150)
plt.show()

## Step 4: Cross-Validation & Best Model Selection

In [ ]:
for name, (model, X_train_use) in models.items():
    cv_scores = cross_val_score(model, X_train_use, y_train, cv=5, scoring='accuracy')
    results[name]['cv_mean'] = cv_scores.mean()
    results[name]['cv_std'] = cv_scores.std()
    print(f'{name}: CV Mean = {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

In [ ]:
best_model = max(results, key=lambda m: results[m]['f1'])
print(f'\nBest model by F1-Score: {best_model}')
print(f'F1-Score: {results[best_model]["f1"]:.4f}')
print(f'Cross-Validation: {results[best_model]["cv_mean"]:.4f} (+/- {results[best_model]["cv_std"]:.4f})')

In [ ]:
with open('../data/model_comparison_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print('Results saved to: ../data/model_comparison_results.pkl')